# Step 3: Aspect-Based Sentiment Analysis (ABSA)
**Role: Person A (Data & ML Engineer)**

Notebook ini berisi **Step 3** dari pipeline proyek:
1. **Load Data**: Membaca `trusted_reviews.csv` hasil Step 2 (Quality Filter)
2. **Step 3a — Aspect Extraction**: Mendeteksi aspek yang disebutkan dalam ulasan (harga, kualitas, pengiriman, layanan) menggunakan keyword matching
3. **Step 3b — Sentiment Classification**: Mengklasifikasikan sentimen per aspek menggunakan model **DistilBERT** (`distilbert-base-uncased-finetuned-sst-2-english`)
4. **Export Output**: Menyimpan hasil lengkap ke `absa_results.csv` untuk dilanjutkan ke Step 4–6 oleh partner

> **Catatan**: Step 4 (Scoring Engine), Step 5 (Decision Engine), dan Step 6 (Visualisasi) dikerjakan oleh partner pada notebook terpisah.

---
## 1. Import Libraries

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Append parent directory untuk import config
sys.path.append(os.path.abspath('..'))
import config

print('Libraries berhasil diimport!')
print(f'Output directory: {config.OUTPUTS_DIR}')

Libraries berhasil diimport!
Output directory: d:\big data\final project\ABSA project\outputs


---
## 2. Load Trusted Reviews
Membaca hasil output Step 2 — hanya ulasan yang telah diverifikasi sebagai `is_trusted = 1`.

In [2]:
trusted_csv_path = os.path.join(config.OUTPUTS_DIR, 'trusted_reviews.csv')
print(f'Membaca: {trusted_csv_path}')

df = pd.read_csv(trusted_csv_path)
print(f'Total ulasan terpercaya: {df.shape[0]:,}')
print(f'Kolom: {list(df.columns)}')
df.head(3)

Membaca: d:\big data\final project\ABSA project\outputs\trusted_reviews.csv
Total ulasan terpercaya: 425,803
Kolom: ['review_id', 'product_id', 'review_body', 'star_rating', 'trust_probability']


,review_id,product_id,review_body,star_rating,trust_probability
0,R1007TU9E6ICBX,B007IHF5QM,"with two caveats, this mount worked great for ...",4,0.747802
1,R100FZKQ3DH330,B001YSAV6A,i bought this because i needed something child...,5,0.680529
2,R100GHF5YMF7F2,B000EPNDEG,i never thought i'd buy an ipod (or any mp3-ty...,4,0.848870


---
## Step 3a: Aspect Extraction (Keyword Matching)
Untuk setiap ulasan, deteksi aspek mana yang disebutkan berdasarkan daftar keyword di `config.ASPECT_KEYWORDS`.

| Aspek | Contoh Keyword |
|---|---|
| harga | expensive, cheap, price, value, worth, cost |
| kualitas | broken, quality, durable, build, defective |
| pengiriman | shipping, delivery, arrived, package, fast |
| layanan | customer service, refund, return, warranty |

In [3]:
ASPECT_KEYWORDS = config.ASPECT_KEYWORDS
ASPECTS = list(ASPECT_KEYWORDS.keys())

def extract_aspects(text):
    """Return dict {aspek: True/False} berdasarkan keyword matching."""
    if not isinstance(text, str):
        return {asp: False for asp in ASPECT_KEYWORDS}
    text_lower = text.lower()
    return {
        asp: any(kw in text_lower for kw in keywords)
        for asp, keywords in ASPECT_KEYWORDS.items()
    }

print('Menjalankan aspect extraction...')
aspect_flags = df['review_body'].apply(extract_aspects)
aspect_df = pd.DataFrame(aspect_flags.tolist())
df = pd.concat([df.reset_index(drop=True), aspect_df], axis=1)

print('Selesai! Distribusi aspek yang terdeteksi:')
for asp in ASPECTS:
    count = df[asp].sum()
    pct = count / len(df) * 100
    print(f'  {asp:12s}: {count:>7,} ulasan ({pct:.1f}%)')

df.head(3)

Menjalankan aspect extraction...
Selesai! Distribusi aspek yang terdeteksi:
  harga       : 177,694 ulasan (41.7%)
  kualitas    : 157,739 ulasan (37.0%)
  pengiriman  :  81,624 ulasan (19.2%)
  layanan     :  77,607 ulasan (18.2%)


,review_id,product_id,review_body,star_rating,trust_probability,harga,kualitas,pengiriman,layanan
0,R1007TU9E6ICBX,B007IHF5QM,"with two caveats, this mount worked great for ...",4,0.747802,False,False,False,False
1,R100FZKQ3DH330,B001YSAV6A,i bought this because i needed something child...,5,0.680529,True,True,False,False
2,R100GHF5YMF7F2,B000EPNDEG,i never thought i'd buy an ipod (or any mp3-ty...,4,0.848870,True,False,False,False


---
## Step 3b: Sentiment Classification per Aspek (DistilBERT)
Menggunakan pretrained model `distilbert-base-uncased-finetuned-sst-2-english` dari HuggingFace untuk mengklasifikasikan sentimen (POSITIVE / NEGATIVE) pada ulasan yang menyebut suatu aspek.

> ⚠️ **Catatan Performa**: Model dijalankan pada sampel maksimal **50.000 ulasan** untuk efisiensi. Proses ini memerlukan waktu beberapa menit (lebih cepat jika menggunakan GPU).

In [5]:
from transformers import pipeline

print('Memuat model DistilBERT dari HuggingFace...')
sentiment_pipeline = pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english',
    truncation=True,
    max_length=512
)
print('Model DistilBERT berhasil dimuat!')

Memuat model DistilBERT dari HuggingFace...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Model DistilBERT berhasil dimuat!


In [7]:
def classify_sentiment_batch(texts, batch_size=64):
    """
    Klasifikasikan sentimen secara batch.
    Return list label: 'POSITIVE', 'NEGATIVE', atau None jika teks tidak valid.
    """
    results = [None] * len(texts)
    valid = [(i, t) for i, t in enumerate(texts) if isinstance(t, str) and len(t.strip()) > 0]
    for i in range(0, len(valid), batch_size):
        batch = valid[i:i + batch_size]
        indices, batch_texts = zip(*batch)
        preds = sentiment_pipeline(list(batch_texts))
        for idx, pred in zip(indices, preds):
            results[idx] = pred['label']
    return results


# Sampling untuk efisiensi (max 50.000 ulasan)
SAMPLE_SIZE = 30000
if len(df) > SAMPLE_SIZE:
    print(f'Dataset besar ({len(df):,} baris). Mengambil sample {SAMPLE_SIZE:,} ulasan...')
    df_sample = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
else:
    df_sample = df.reset_index(drop=True)
    print(f'Menggunakan seluruh dataset: {len(df_sample):,} ulasan')

# Klasifikasikan sentimen per aspek
print('\nMemulai klasifikasi sentimen per aspek...')
for asp in ASPECTS:
    print(f'  [{asp}] memproses...')
    mask = df_sample[asp] == True
    texts = df_sample.loc[mask, 'review_body'].tolist()

    if texts:
        labels = classify_sentiment_batch(texts)
        df_sample.loc[mask, f'sentiment_{asp}'] = labels

    # Ulasan yang tidak menyebut aspek ini → N/A
    if f'sentiment_{asp}' not in df_sample.columns:
        df_sample[f'sentiment_{asp}'] = 'N/A'
    else:
        df_sample[f'sentiment_{asp}'] = df_sample[f'sentiment_{asp}'].fillna('N/A')

    pos = (df_sample[f'sentiment_{asp}'] == 'POSITIVE').sum()
    neg = (df_sample[f'sentiment_{asp}'] == 'NEGATIVE').sum()
    na  = (df_sample[f'sentiment_{asp}'] == 'N/A').sum()
    print(f'    POSITIVE: {pos:,} | NEGATIVE: {neg:,} | N/A: {na:,}')

print('\nKlasifikasi sentimen selesai!')
df_sample.head(3)

Dataset besar (425,803 baris). Mengambil sample 30,000 ulasan...

Memulai klasifikasi sentimen per aspek...
  [harga] memproses...
    POSITIVE: 6,571 | NEGATIVE: 6,015 | N/A: 0
  [kualitas] memproses...
    POSITIVE: 6,087 | NEGATIVE: 5,051 | N/A: 0
  [pengiriman] memproses...
    POSITIVE: 2,484 | NEGATIVE: 3,287 | N/A: 0
  [layanan] memproses...
    POSITIVE: 1,448 | NEGATIVE: 3,883 | N/A: 0

Klasifikasi sentimen selesai!


,review_id,product_id,review_body,star_rating,trust_probability,harga,kualitas,pengiriman,layanan,sentiment_harga,sentiment_kualitas,sentiment_pengiriman,sentiment_layanan
0,R366Y8RH9NLUOD,B004GV3AZ6,i bought 2 of these and they both worked great...,4,0.512622,False,False,False,False,nan,nan,nan,nan
1,R19WX2VCQUBRDU,B00BIOMUIW,i'm using this on my at-120 body. it'll fit th...,4,0.614660,False,False,False,False,nan,nan,nan,nan
2,RYBO7ZXWHMXFQ,B005FVNGZK,we have had a number of these clips. the newe...,3,0.592101,True,False,False,True,NEGATIVE,nan,nan,NEGATIVE


---
## 4. Export Output untuk Step 4–6 (Partner)
Menyimpan hasil ABSA lengkap ke file `absa_results.csv`. File ini akan menjadi **input utama** untuk partner yang mengerjakan Step 4 (Scoring), Step 5 (Decision), dan Step 6 (Visualisasi).

Kolom yang diekspor:
- `review_id`, `product_id`, `review_body`, `star_rating`, `trust_probability`
- `harga`, `kualitas`, `pengiriman`, `layanan` → True/False (aspek terdeteksi?)
- `sentiment_harga`, `sentiment_kualitas`, `sentiment_pengiriman`, `sentiment_layanan` → POSITIVE / NEGATIVE / N/A

In [8]:
# Tentukan kolom yang diekspor
base_cols = ['review_id', 'product_id', 'review_body', 'star_rating', 'trust_probability']
aspect_flag_cols = ASPECTS                                    # True/False per aspek
sentiment_cols   = [f'sentiment_{asp}' for asp in ASPECTS]   # POSITIVE/NEGATIVE/N/A

export_cols = base_cols + aspect_flag_cols + sentiment_cols
df_export = df_sample[export_cols]

output_path = os.path.join(config.OUTPUTS_DIR, 'absa_results.csv')
df_export.to_csv(output_path, index=False, encoding='utf-8')

print(f'File berhasil disimpan ke: {output_path}')
print(f'Ukuran file output: {df_export.shape}')
print(f'\nDistribusi sentimen per aspek:')
for asp in ASPECTS:
    col = f'sentiment_{asp}'
    dist = df_export[col].value_counts()
    print(f'  {asp:12s}: {dict(dist)}')

print('\n=== Step 3 selesai! ===')
print('Output siap diserahkan ke partner untuk Step 4-6.')
df_export.head(5)

File berhasil disimpan ke: d:\big data\final project\ABSA project\outputs\absa_results.csv
Ukuran file output: (30000, 13)

Distribusi sentimen per aspek:
  harga       : {np.str_('nan'): np.int64(17414), 'POSITIVE': np.int64(6571), 'NEGATIVE': np.int64(6015)}
  kualitas    : {np.str_('nan'): np.int64(18862), 'POSITIVE': np.int64(6087), 'NEGATIVE': np.int64(5051)}
  pengiriman  : {np.str_('nan'): np.int64(24229), 'NEGATIVE': np.int64(3287), 'POSITIVE': np.int64(2484)}
  layanan     : {np.str_('nan'): np.int64(24669), 'NEGATIVE': np.int64(3883), 'POSITIVE': np.int64(1448)}

=== Step 3 selesai! ===
Output siap diserahkan ke partner untuk Step 4-6.


,review_id,product_id,review_body,star_rating,trust_probability,harga,kualitas,pengiriman,layanan,sentiment_harga,sentiment_kualitas,sentiment_pengiriman,sentiment_layanan
0,R366Y8RH9NLUOD,B004GV3AZ6,i bought 2 of these and they both worked great...,4,0.512622,False,False,False,False,nan,nan,nan,nan
1,R19WX2VCQUBRDU,B00BIOMUIW,i'm using this on my at-120 body. it'll fit th...,4,0.614660,False,False,False,False,nan,nan,nan,nan
2,RYBO7ZXWHMXFQ,B005FVNGZK,we have had a number of these clips. the newe...,3,0.592101,True,False,False,True,NEGATIVE,nan,nan,NEGATIVE
3,R240OIPQEIK0RY,B0063705PE,during the digital tv transition here in the u...,5,0.902345,True,True,False,True,POSITIVE,POSITIVE,nan,POSITIVE
4,R2YZJB5981DI2B,B000UV4EUG,when i first ordered this product i had such h...,1,0.838424,True,True,False,False,NEGATIVE,NEGATIVE,nan,nan
